In [ ]:
import numpy as np
from qutip import Bloch
import numba
from numba import njit, prange
import pickle
import os
import shutil
import imageio
from IPython.display import Image, display
from multiprocessing import Pool, cpu_count
from functools import partial
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:

sz = np.array(([[1,0], [0,-1]]), dtype=complex); sx = np.array(([[0,1],[1,0]]), dtype=complex); sy = np.array(([[0,-1j],[1j,0]]), dtype=complex) 


In [ ]:
# ======================================
# Example to choose the Axis orientation
# ======================================

# ! Comment matplotlib.use('Agg') and restar kernell to run !

b = Bloch()
b.view = [-60, 30]  # Azimuthal angle, High form xy plane

b.show()

In [ ]:
# -----------------------
# Directory and file name
# -----------------------
input_path = "../Results/Data/"
input_file = os.path.join(input_path, "results_General_prova_diff.pkl") # change for different plot

output_path = "../Results/Bloch_Sphere"
gif_path = os.path.join(output_path, 'General_prova_prob.gif')   # change : name gif
os.makedirs(output_path, exist_ok=True)

# --------------------
# Temporary Directory
# --------------------
temp_dir = os.path.join(output_path, 'temp_bloch_frames')

if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
os.makedirs(temp_dir)

In [ ]:
# ------------
# Load Results
# ------------
with open(input_file, 'rb') as f:
    results = pickle.load(f)

# Extract and sample data (every 10 steps)
sample_step = 10 

v_x = results[0.01][1000]['trajectory_wf']['bloch']['samples_x'][::sample_step, 0]
v_y = results[0.01][1000]['trajectory_wf']['bloch']['samples_y'][::sample_step, 0]
v_z = results[0.01][1000]['trajectory_wf']['bloch']['samples_z'][::sample_step, 0]
avg_x = results[0.01][1000]['trajectory_wf']['bloch']['avg_x'][::sample_step]
avg_y = results[0.01][1000]['trajectory_wf']['bloch']['avg_y'][::sample_step]
avg_z = results[0.01][1000]['trajectory_wf']['bloch']['avg_z'][::sample_step]

In [ ]:
def generate_single_frame(i, v_x, v_y, v_z, avg_x, avg_y, avg_z, temp_dir, n_steps):
    """
    Generate a single frame of the Bloch sphere animation.
    """
    try:        
        sphere = Bloch()
        sphere.view = [-60, 30]
            
        # Add points with fading effect
        for j in range(i + 1):    # j=0 fisrts point, till last point in time i+1, with fading exponentialy increasing
            sphere.add_points([v_x[j], v_y[j], v_z[j]], alpha = np.exp((j - i) / 15))
            
        # Add average vector
        sphere.add_vectors([avg_x[i], avg_y[i], avg_z[i]])
            
        # Configuration
        sphere.point_color = 'b'
        sphere.point_marker = 'o'
        sphere.point_size = [25]
            
        # Generate and save
        sphere.make_sphere()
        frame_name = f'bloch_{str(i).zfill(len(str(n_steps)))}.png'
        frame_path = os.path.join(temp_dir, frame_name)
        sphere.fig.savefig(frame_path, dpi=80, bbox_inches='tight')
        plt.close(sphere.fig)
            
        return frame_path

    except Exception as e:
        print(f"Error on frame {i}: {e}")
        return None

n_steps = len(v_x) # number of time steps
n_processes = 12 # number of CPU

print(f"Total frames to generate: {n_steps} (subsampled by factor {step})")
print(f"Using {n_processes} parallel processes")
print(f"Generating frames...")

# Parallel frame generation
generate_frame_partial = partial( generate_single_frame, v_x=v_x, v_y=v_y, v_z=v_z,
    avg_x=avg_x, avg_y=avg_y, avg_z=avg_z, temp_dir=temp_dir, n_steps=n_steps)

with Pool(processes=n_processes) as pool:
    frame_paths = pool.map(generate_frame_partial, range(n_steps))


# Create GIF 
print("Creating GIF...")
filenames = sorted([os.path.join(temp_dir, f) for f in os.listdir(temp_dir) if f.endswith('.png')])

with imageio.get_writer(gif_path, mode='I', duration=0.1) as writer:   # duration of gif steps
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

# Clean up temporary files
shutil.rmtree(temp_dir)

# Display result
file_size_mb = os.path.getsize(gif_path) / (1024 * 1024)
print(f"\nGIF saved to: {gif_path}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"Duration: {n_steps * 0.05:.2f} seconds")
print(f"Temporary files cleaned up")


In [ ]:
display(Image(filename=gif_path))